# PRISM Distribution Validation

Validates `PRP_1000_full_pretreatment.xlsx` (1000 rows, 44 cols) against
expected distributions from the GenRocket review PDF.

**Pass/Fail Criteria:**
- Binary flags with range target: PASS if observed % within range
- Binary flags with point target: PASS if within ±2.5pp
- Bucketed distributions: PASS if max absolute bucket diff ≤ 10pp
- Categorical: PASS if distribution shape roughly matches (≤ 10pp)


In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_excel('PRP_1000_full_pretreatment.xlsx')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

results = []

## Helper Functions

In [ ]:
def check_binary_flag(col, low, high):
    """Check binary flag proportion against expected range."""
    if col not in df.columns:
        return {'Variable': col, 'Expected_Distribution': f'{low*100:.1f}%-{high*100:.1f}%',
                'Observed_Distribution': 'COLUMN NOT FOUND', 'Max_Bucket_Diff': None,
                'Avg_Bucket_Diff': None, 'KD_Result': 'FAIL - Missing Column'}
    obs_pct = df[col].mean()
    passed = low <= obs_pct <= high
    diff = 0 if passed else min(abs(obs_pct - low), abs(obs_pct - high))
    print(f'{col}: observed={obs_pct*100:.1f}%, expected={low*100:.1f}%-{high*100:.1f}% -> {"PASS" if passed else "FAIL"}')
    return {'Variable': col, 'Expected_Distribution': f'{low*100:.1f}%-{high*100:.1f}% of 1s',
            'Observed_Distribution': f'{obs_pct*100:.1f}% of 1s',
            'Max_Bucket_Diff': round(diff * 100, 2),
            'Avg_Bucket_Diff': round(diff * 100, 2),
            'KD_Result': 'PASS' if passed else 'FAIL'}

def check_bucketed(col, buckets, bucket_fn, expected_pcts):
    """Check bucketed distribution against expected percentages."""
    if col not in df.columns:
        return {'Variable': col, 'Expected_Distribution': str(dict(zip(buckets, expected_pcts))),
                'Observed_Distribution': 'COLUMN NOT FOUND', 'Max_Bucket_Diff': None,
                'Avg_Bucket_Diff': None, 'KD_Result': 'FAIL - Missing Column'}
    bucketed = bucket_fn(df[col])
    obs_counts = bucketed.value_counts(normalize=True)
    obs_pcts = [obs_counts.get(b, 0) * 100 for b in buckets]
    diffs = [abs(o - e) for o, e in zip(obs_pcts, expected_pcts)]
    max_diff = max(diffs)
    avg_diff = sum(diffs) / len(diffs)
    passed = max_diff <= 10.0
    exp_str = ', '.join([f'{b}: {e:.0f}%' for b, e in zip(buckets, expected_pcts)])
    obs_str = ', '.join([f'{b}: {o:.1f}%' for b, o in zip(buckets, obs_pcts)])
    print(f'{col}: max_diff={max_diff:.1f}pp -> {"PASS" if passed else "FAIL"}')
    print(f'  Expected: {exp_str}')
    print(f'  Observed: {obs_str}')
    return {'Variable': col, 'Expected_Distribution': exp_str,
            'Observed_Distribution': obs_str,
            'Max_Bucket_Diff': round(max_diff, 2),
            'Avg_Bucket_Diff': round(avg_diff, 2),
            'KD_Result': 'PASS' if passed else 'FAIL'}

def check_categorical(col, expected_dict, tolerance=10.0):
    """Check categorical distribution against expected percentages."""
    if col not in df.columns:
        return {'Variable': col, 'Expected_Distribution': str(expected_dict),
                'Observed_Distribution': 'COLUMN NOT FOUND', 'Max_Bucket_Diff': None,
                'Avg_Bucket_Diff': None, 'KD_Result': 'FAIL - Missing Column'}
    obs_counts = df[col].value_counts(normalize=True) * 100
    diffs = [abs(obs_counts.get(cat, 0) - exp) for cat, exp in expected_dict.items()]
    max_diff = max(diffs) if diffs else 0
    avg_diff = sum(diffs) / len(diffs) if diffs else 0
    passed = max_diff <= tolerance
    exp_str = ', '.join([f'{k}: {v:.0f}%' for k, v in expected_dict.items()])
    obs_str = ', '.join([f'{k}: {v:.1f}%' for k, v in obs_counts.head(10).items()])
    print(f'{col}: max_diff={max_diff:.1f}pp -> {"PASS" if passed else "FAIL"}')
    print(f'  Expected: {exp_str}')
    print(f'  Observed: {obs_str}')
    return {'Variable': col, 'Expected_Distribution': exp_str,
            'Observed_Distribution': obs_str,
            'Max_Bucket_Diff': round(max_diff, 2),
            'Avg_Bucket_Diff': round(avg_diff, 2),
            'KD_Result': 'PASS' if passed else 'FAIL'}

## Bucketed Distribution Variables

In [ ]:
# 1. admits_last_6m: 0: 85%, 1: 10%, 2: 3%, 3+: 2%
results.append(check_bucketed('admits_last_6m',
    ['0', '1', '2', '3+'],
    lambda s: s.clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3+'}),
    [85, 10, 3, 2]))

# 2. age: 18-34: 20%, 35-49: 25%, 50-64: 35%, 65+: 20%
results.append(check_bucketed('age',
    ['18-34', '35-49', '50-64', '65+'],
    lambda s: pd.cut(s, bins=[17, 34, 49, 64, 200], labels=['18-34', '35-49', '50-64', '65+']),
    [20, 25, 35, 20]))

# 15. ed_visits_last_30d: 0: 92%, 1: 6%, 2: 1.5%, 3+: 0.5%
results.append(check_bucketed('ed_visits_last_30d',
    ['0', '1', '2', '3+'],
    lambda s: s.clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3+'}),
    [92, 6, 1.5, 0.5]))

# 16. ed_visits_last_6m: 0: 70%, 1: 15%, 2: 8%, 3+: 7%
results.append(check_bucketed('ed_visits_last_6m',
    ['0', '1', '2', '3+'],
    lambda s: s.clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3+'}),
    [70, 15, 8, 7]))

# 22. living_alone_flag: 0: 74%, 1: 23%, Unknown/Missing: 3%
results.append(check_bucketed('living_alone_flag',
    ['0', '1', 'Unknown/Missing'],
    lambda s: s.astype(str).map(lambda x: '0' if x in ('0','0.0') else ('1' if x in ('1','1.0') else 'Unknown/Missing')),
    [74, 23, 3]))

# 23. med_adherence_pdc: <0.5: 10%, 0.5-0.79: 30%, >=0.80: 60%
results.append(check_bucketed('med_adherence_pdc',
    ['<0.5', '0.5-0.79', '>=0.80'],
    lambda s: pd.cut(s, bins=[-0.01, 0.5, 0.79, 1.01], labels=['<0.5', '0.5-0.79', '>=0.80'], right=False),
    [10, 30, 60]))

# 24. observation_stays_last_6m: 0: 91%, 1: 6%, 2: 2%, 3+: 1%
results.append(check_bucketed('observation_stays_last_6m',
    ['0', '1', '2', '3+'],
    lambda s: s.clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3+'}),
    [91, 6, 2, 1]))

# 26. pcp_visits_last_6m: 0: 15%, 1-2: 30%, 3-5: 35%, 6+: 20%
results.append(check_bucketed('pcp_visits_last_6m',
    ['0', '1-2', '3-5', '6+'],
    lambda s: pd.cut(s, bins=[-1, 0, 2, 5, 9999], labels=['0', '1-2', '3-5', '6+']),
    [15, 30, 35, 20]))

# 32. rx_count_last_6m: 0: 10%, 1-2: 12%, 3-5: 18%, 6-11: 25%, 12-23: 22%, 24-35: 8%, 36+: 5%
results.append(check_bucketed('rx_count_last_6m',
    ['0', '1-2', '3-5', '6-11', '12-23', '24-35', '36+'],
    lambda s: pd.cut(s, bins=[-1, 0, 2, 5, 11, 23, 35, 9999], labels=['0', '1-2', '3-5', '6-11', '12-23', '24-35', '36+']),
    [10, 12, 18, 25, 22, 8, 5]))

# 34. specialist_visits_last_6m: 0: 30%, 1-2: 30%, 3-5: 25%, 6+: 15%
results.append(check_bucketed('specialist_visits_last_6m',
    ['0', '1-2', '3-5', '6+'],
    lambda s: pd.cut(s, bins=[-1, 0, 2, 5, 9999], labels=['0', '1-2', '3-5', '6+']),
    [30, 30, 25, 15]))

# 36. total_cost_last_6m
results.append(check_bucketed('total_cost_last_6m',
    ['$0-499', '$500-1999', '$2000-4999', '$5000-9999', '$10000-24999', '$25000-49999', '$50000+'],
    lambda s: pd.cut(s, bins=[-1, 499, 1999, 4999, 9999, 24999, 49999, 999999999],
                     labels=['$0-499', '$500-1999', '$2000-4999', '$5000-9999', '$10000-24999', '$25000-49999', '$50000+']),
    [10, 20, 25, 20, 15, 7, 3]))

## Score Variables (0-100 range)

In [ ]:
def bucket_score(s):
    return pd.cut(s, bins=[-1, 19, 39, 59, 79, 100], labels=['0-19', '20-39', '40-59', '60-79', '80-100'])

# 39. current_risk_score: 0-19: 8%, 20-39: 22%, 40-59: 35%, 60-79: 25%, 80-100: 10%
results.append(check_bucketed('current_risk_score',
    ['0-19', '20-39', '40-59', '60-79', '80-100'],
    bucket_score, [8, 22, 35, 25, 10]))

# 40. percolator_clinical_score: 0-19: 12%, 20-39: 28%, 40-59: 30%, 60-79: 20%, 80-100: 10%
results.append(check_bucketed('percolator_clinical_score',
    ['0-19', '20-39', '40-59', '60-79', '80-100'],
    bucket_score, [12, 28, 30, 20, 10]))

# 41. percolator_sdoh_score: 0-19: 35%, 20-39: 25%, 40-59: 20%, 60-79: 12%, 80-100: 8%
results.append(check_bucketed('percolator_sdoh_score',
    ['0-19', '20-39', '40-59', '60-79', '80-100'],
    bucket_score, [35, 25, 20, 12, 8]))

# 42. percolator_utilization_score: 0-19: 35%, 20-39: 30%, 40-59: 18%, 60-79: 10%, 80-100: 7%
results.append(check_bucketed('percolator_utilization_score',
    ['0-19', '20-39', '40-59', '60-79', '80-100'],
    bucket_score, [35, 30, 18, 10, 7]))

## Binary Flag Variables

In [ ]:
# 3. anxiety_flag: 18%-28%
results.append(check_binary_flag('anxiety_flag', 0.18, 0.28))

# 4. asthma_flag: 12%-18%
results.append(check_binary_flag('asthma_flag', 0.12, 0.18))

# 5. behavioral_health_risk_flag: 20%-30%
results.append(check_binary_flag('behavioral_health_risk_flag', 0.20, 0.30))

# 7. chf_flag: 14% (range 11.5%-16.5%)
results.append(check_binary_flag('chf_flag', 0.115, 0.165))

# 8. ckd_flag: 15% (range 12.5%-17.5%)
results.append(check_binary_flag('ckd_flag', 0.125, 0.175))

# 10. copd_flag: 11% (range 8.5%-13.5%)
results.append(check_binary_flag('copd_flag', 0.085, 0.135))

# 12. depression_flag: 20%-30%
results.append(check_binary_flag('depression_flag', 0.20, 0.30))

# 13. diabetes_flag: 28% (range 25.5%-30.5%)
results.append(check_binary_flag('diabetes_flag', 0.255, 0.305))

# 14. dual_eligible: 18%-25%
results.append(check_binary_flag('dual_eligible', 0.18, 0.25))

# 17. food_insecurity_flag: 15%-25%
results.append(check_binary_flag('food_insecurity_flag', 0.15, 0.25))

# 19. high_cost_drug_flag: 5%-10%
results.append(check_binary_flag('high_cost_drug_flag', 0.05, 0.10))

# 20. housing_instability_flag: 8%-15%
results.append(check_binary_flag('housing_instability_flag', 0.08, 0.15))

# 25. opioid_flag: 5%-10%
results.append(check_binary_flag('opioid_flag', 0.05, 0.10))

# 28. polypharmacy_flag: 15%-25%
results.append(check_binary_flag('polypharmacy_flag', 0.15, 0.25))

# 29. pregnancy_flag: 3%-6%
results.append(check_binary_flag('pregnancy_flag', 0.03, 0.06))

# 35. substance_use_flag: 8%-15%
results.append(check_binary_flag('substance_use_flag', 0.08, 0.15))

# 37. transportation_barrier_flag: 12%-20%
results.append(check_binary_flag('transportation_barrier_flag', 0.12, 0.20))

# 38. utilities_insecurity_flag: 12% (range 9.5%-14.5%)
results.append(check_binary_flag('utilities_insecurity_flag', 0.095, 0.145))

## Categorical Variables

In [ ]:
# 6. case_manager_name: ~4-5 managers with 20-24% each, ~5% unassigned
obs_cm = df['case_manager_name'].value_counts(normalize=True) * 100
top_val = obs_cm.iloc[0]
n_cats = len(obs_cm)
passed_cm = (top_val <= 30) and (n_cats >= 4)
max_diff_cm = max(0, top_val - 24)
obs_str_cm = ', '.join([f'{k}: {v:.1f}%' for k, v in obs_cm.head(6).items()])
print(f'case_manager_name: {n_cats} managers, top={top_val:.1f}% -> {"PASS" if passed_cm else "FAIL"}')
print(f'  Observed: {obs_str_cm}')
results.append({'Variable': 'case_manager_name',
    'Expected_Distribution': '~4-5 managers at 20-24% each, ~5% unassigned',
    'Observed_Distribution': obs_str_cm,
    'Max_Bucket_Diff': round(max_diff_cm, 2),
    'Avg_Bucket_Diff': round(max_diff_cm / 2, 2),
    'KD_Result': 'PASS' if passed_cm else 'FAIL'})

# 9. client_contract
results.append(check_categorical('client_contract', {
    'Medicaid MCO': 40, 'Medicaid Expansion': 18, 'Dual Eligibility': 22,
    'MLTSS': 8, 'Behavioral Health': 6, 'Health Home': 3,
    'Maternal Health': 2, 'SUD': 1}))

# 11. county
obs_county = df['county'].value_counts(normalize=True) * 100
top_county = obs_county.iloc[0]
n_counties = len(obs_county)
passed_cty = (18 <= top_county <= 38) and (n_counties >= 8)
max_diff_cty = abs(top_county - 28)
obs_str_cty = ', '.join([f'{k}: {v:.1f}%' for k, v in obs_county.head(10).items()])
print(f'county: {n_counties} counties, top={top_county:.1f}% -> {"PASS" if passed_cty else "FAIL"}')
results.append({'Variable': 'county',
    'Expected_Distribution': 'Top: 28%, 18%, 14%, 10%, 8%, 6%, 5%, 4%, 2%, 5% unknown',
    'Observed_Distribution': obs_str_cty,
    'Max_Bucket_Diff': round(max_diff_cty, 2),
    'Avg_Bucket_Diff': round(max_diff_cty / 2, 2),
    'KD_Result': 'PASS' if passed_cty else 'FAIL'})

# 18. gender: Female: 56%, Male: 43%
results.append(check_categorical('gender', {'Female': 56, 'Male': 43}, tolerance=10.0))

# 21. language: English: 78%, Spanish: 13%
results.append(check_categorical('language', {'English': 78, 'Spanish': 13}, tolerance=10.0))

# 27. plan_type: Comprehensive_MCO: 70%, MLTSS: 8%, D-SNP: 7%
results.append(check_categorical('plan_type', {'Comprehensive_MCO': 70, 'MLTSS': 8, 'D-SNP': 7}, tolerance=10.0))

# 30. program
results.append(check_categorical('program', {
    'Complex Care Management': 30, 'High_Risk Case Management': 20,
    'Behavioral Health Integration': 14, 'Health Home': 10, 'LTSS': 8}, tolerance=10.0))

# 31. risk_tier
results.append(check_categorical('risk_tier', {
    'Low': 25, 'Moderate': 35, 'High': 25, 'Very High': 10,
    'Rising Risk': 3, 'Impactable High Risk': 2}, tolerance=10.0))

# 33. service_region
results.append(check_categorical('service_region', {
    'Urban Core': 45, 'Suburban': 25, 'Rural': 20, 'Frontier': 5}, tolerance=10.0))

## Summary & Export

In [ ]:
summary_df = pd.DataFrame(results)
print(f'Total variables checked: {len(summary_df)}')
print(f'PASS: {(summary_df["KD_Result"] == "PASS").sum()}')
print(f'FAIL: {(summary_df["KD_Result"] != "PASS").sum()}')
print()
print(summary_df[['Variable', 'Max_Bucket_Diff', 'KD_Result']].to_string(index=False))

# Export to Excel
summary_df.to_excel('distribution_validation_results.xlsx', index=False, sheet_name='Distribution Validation')
print(f'\nResults exported to: distribution_validation_results.xlsx')

## Full Results Table

In [ ]:
summary_df.style.applymap(
    lambda v: 'background-color: #d4edda' if v == 'PASS' else ('background-color: #f8d7da' if v == 'FAIL' else ''),
    subset=['KD_Result'])